In [1]:
%load_ext autoreload
%autoreload 2

# Import

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from typing import Union
from glob import glob

import sys
PHASE1_SRC = Path("../Phase1/src/")
if str(PHASE1_SRC) not in sys.path:
    sys.path.insert(0, str(PHASE1_SRC))

from space import Space, GFLOWNET_ENV
from reward import reward_peak, reward_latent
from machinelearning import train_single_model, predict_with_model, preprocess, build_models
from VEM import oracle_fn, TARGET_NAMES
from plotting import *

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from umap import UMAP
from sklearn.decomposition import PCA


# Loading

In [29]:
run = "grid_results_5"

In [30]:
grid = pd.read_csv(f"{run}/grid_summary.csv")
REWARD_FILTER = "reward_peak" # "reward_latent" | "reward_peak"
grid = grid[grid["reward_fn_name"] == REWARD_FILTER]

In [31]:
grid.head()

,run,status,elapsed_sec,best_reward,mean_reward,total_oracle_budget,sampling_strategy,acquisition,reward_fn_name,seed,init_method,n_init,n_candidates_per_iter,n_iterations,gfn_loss,gfn_gflownet,gfn_policy,gfn_batch_size,gfn_n_train_steps
0,run_0,ok,152.2,0.527267,0.165545,350,gflownet,top_k,reward_peak,123,latin_hypercube,50,20,15,detailedbalance,detailedbalance,mlp_detailedbalance,100,100
1,run_1,ok,148.1,0.691377,0.144556,350,gflownet,top_k,reward_peak,123,latin_hypercube,200,10,15,detailedbalance,detailedbalance,mlp_detailedbalance,100,100
2,run_2,ok,148.3,0.572327,0.138868,350,gflownet,top_k,reward_peak,123,latin_hypercube,275,5,15,detailedbalance,detailedbalance,mlp_detailedbalance,100,100
3,run_3,ok,144.8,0.765969,0.166818,350,gflownet,top_k,reward_peak,123,latin_hypercube,50,20,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100
4,run_4,ok,145.0,0.606062,0.143531,350,gflownet,top_k,reward_peak,123,latin_hypercube,200,10,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100


In [32]:
from glob import glob
paths = sorted(glob(f"{run}/*/al_metrics.csv"))
paths = [p for p in paths if Path(p).parent.name in grid["run"].values] # filters per reward as wreitten above
if not paths:
    raise ValueError("No files found at al_results/*/al_metrics.csv")
df = pd.concat([load(p).assign(run=Path(p).parent.name) for p in paths], ignore_index=True)

In [33]:
df

,iteration,time,n_samples,oracle_evals_total,oracle_evals_this_iter,best_reward_so_far,mean_reward_so_far,proxy_r2,proxy_mae,sampling_strategy,best_reward_this_iter,mean_reward_this_iter,best_predicted_reward,elapsed_sec,run
0,1,12:14:09,70,70,20,0.400640,0.114011,0.255804,0.091992,gflownet,0.264725,0.121976,0.469609,21.5,run_0
1,2,12:14:19,90,90,20,0.487418,0.143619,0.529564,0.096624,gflownet,0.487418,0.247248,0.441824,10.0,run_0
2,3,12:14:28,110,110,20,0.487418,0.142826,0.589719,0.068975,gflownet,0.324321,0.139253,0.398895,9.1,run_0
3,4,12:14:37,130,130,20,0.487418,0.151749,0.527700,0.069339,gflownet,0.420164,0.200829,0.368630,9.0,run_0
4,5,12:14:46,150,150,20,0.487418,0.151029,0.601306,0.063257,gflownet,0.455140,0.146345,0.311618,9.1,run_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4855,11,12:37:41,270,270,20,0.672271,0.155802,0.561178,0.062411,gflownet,0.432920,0.199069,0.357028,9.7,run_9
4856,12,12:37:50,290,290,20,0.672271,0.156801,0.506546,0.068046,gflownet,0.438975,0.170290,0.500711,9.2,run_9
4857,13,12:37:59,310,310,20,0.672271,0.161086,0.569408,0.059301,gflownet,0.466767,0.223229,0.461926,9.2,run_9
4858,14,12:38:09,330,330,20,0.672271,0.163583,0.609495,0.060351,gflownet,0.558547,0.202282,0.389880,9.4,run_9


In [34]:
df[df["sampling_strategy"]=="gflownet"].groupby("run").count().count().iloc[0]

144

In [35]:
best_runs(df)

,run,final_mean_reward,final_best_reward,final_proxy_r2,final_proxy_mae,total_oracle_evals,sampling_strategy
0,run_616,0.169895,0.898810,0.577924,0.067356,350,genetic
1,run_53,0.128209,0.852505,0.643067,0.068648,350,gflownet
2,run_557,0.123533,0.779775,0.637671,0.073102,350,gp
3,run_435,0.159771,0.775087,0.577851,0.056669,350,grid
4,run_375,0.152833,0.815178,0.519941,0.070699,350,lhs
5,run_338,0.126163,0.778357,0.640593,0.068354,350,random


## Visualization

In [36]:
# filter out elapsed_sec and sampling_strategy fom grid before merging 
metrics = df.merge(
    grid[[c for c in grid.columns if c not in ["elapsed_sec", "sampling_strategy"]]],
    left_on="run",
    right_on="run",
    how="left",
)

In [37]:
select_best_runs(metrics, metric="best_reward")

,iteration,time,n_samples,oracle_evals_total,oracle_evals_this_iter,best_reward_so_far,mean_reward_so_far,proxy_r2,proxy_mae,sampling_strategy,...,seed,init_method,n_init,n_candidates_per_iter,n_iterations,gfn_loss,gfn_gflownet,gfn_policy,gfn_batch_size,gfn_n_train_steps
4455,1,00:22:35,210,210,10,0.595374,0.113315,0.600383,0.066662,genetic,...,123,random,200,10,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100
4456,2,00:22:36,220,220,10,0.595374,0.119158,0.639473,0.070994,genetic,...,123,random,200,10,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100
4457,3,00:22:36,230,230,10,0.632773,0.125051,0.560939,0.072631,genetic,...,123,random,200,10,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100
4458,4,00:22:36,240,240,10,0.632773,0.132000,0.623233,0.075079,genetic,...,123,random,200,10,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100
4459,5,00:22:36,250,250,10,0.632773,0.136992,0.578756,0.066723,genetic,...,123,random,200,10,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1990,11,00:06:30,330,330,5,0.778357,0.120173,0.627656,0.071741,random,...,789,latin_hypercube,275,5,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100
1991,12,00:06:30,335,335,5,0.778357,0.123674,0.604440,0.070866,random,...,789,latin_hypercube,275,5,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100
1992,13,00:06:30,340,340,5,0.778357,0.125195,0.673931,0.070516,random,...,789,latin_hypercube,275,5,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100
1993,14,00:06:30,345,345,5,0.778357,0.125002,0.685446,0.069211,random,...,789,latin_hypercube,275,5,15,trajectorybalance,trajectorybalance,mlp_trajectorybalance,100,100


In [38]:
plot_reward_curves(select_best_runs(metrics, metric="mean_reward"), metric="mean_reward_this_iter", title="Mean reward per iteration", ceiling=True)

In [39]:
aggregate_seeds(metrics, "mean_reward_this_iter", groupby=("sampling_strategy",)).groupby(["sampling_strategy", "iteration"]).agg(
    mean=("mean", "mean"),
    std=("std", "mean"),
    sem=("sem", "mean"),
    ci95=("ci95", "mean"),
)

mean       std       sem      ci95
sampling_strategy iteration                                        
genetic           1          0.182313  0.043131  0.017608  0.034512
                  2          0.206249  0.060144  0.024554  0.048125
                  3          0.211230  0.054096  0.022084  0.043285
                  4          0.220232  0.044926  0.018341  0.035948
                  5          0.211138  0.034998  0.014288  0.028004
...                               ...       ...       ...       ...
random            11         0.190571  0.043163  0.017621  0.034537
                  12         0.189061  0.033938  0.013855  0.027156
                  13         0.195966  0.036478  0.014892  0.029188
                  14         0.185908  0.042310  0.017273  0.033855
                  15         0.194739  0.026607  0.010862  0.021290

[90 rows x 4 columns]

In [40]:
plot_seed_variance(metrics, metric="mean_reward_this_iter")

In [41]:
plot_oracle_efficiency(metrics, metric="mean_reward_this_iter")

In [42]:
plot_proxy_curves(metrics)

In [43]:
plot_final_boxplot(metrics, metric="best_reward")

In [45]:
plot_reward_heatmap(metrics, metric="mean_reward")

In [51]:
plot_budget_tradeoff(metrics, metric="mean_reward")

In [52]:
embedding, preprocessor, reducer = compute_global_embedding(
    method="umap"
)

/u/porchetv/Desktop/PhD/plasma/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [53]:
sampled_dict_best = {}
for samp in df["sampling_strategy"].unique():
    run_id = best_runs(df).loc[best_runs(df)["sampling_strategy"] == samp, "run"].values[0]
    run_df = pd.read_csv(f"{run}/{run_id}/dataset.csv")
    sampled_dict_best[samp] = prepare_run_embedding(run_df, preprocessor, reducer)

In [54]:
plot_sampling_grid(sampled_dict_best, cols=2)

In [55]:
plot_exploration_dashboard(sampled_dict_best, preprocessor)

In [56]:
plot_peak_coverage(sampled_dict_best, "inverse")

In [57]:
plot_peak_coverage_heatmap(sampled_dict_best)

In [58]:
sampled_dict_all = {}
for samp in df["sampling_strategy"].unique():
    run_ids = df.loc[df["sampling_strategy"] == samp, "run"].unique()
    pooled = pd.concat(
        [pd.read_csv(f"{run}/{r}/dataset.csv") for r in run_ids],
        ignore_index=True,
    )
    sampled_dict_all[samp] = prepare_run_embedding(pooled, preprocessor, reducer)
    

In [62]:
plot_exploration_dashboard(sampled_dict_all, preprocessor)

In [61]:
plot_peak_coverage(sampled_dict_all, 'inverse')

In [60]:
plot_peak_coverage_heatmap(sampled_dict_all)

In [59]:
df = metrics[metrics["iteration"] > 0]


# plot
fig = go.Figure()
for i, val in enumerate(sorted(df["gfn_loss"].dropna().unique())):
    agg = df[df["gfn_loss"] == val].groupby("iteration")["mean_reward_this_iter"].agg(["mean","std"]).reset_index()
    c = f"hsl({i * 60}, 70%, 50%)"
    fig.add_trace(go.Scatter(x=agg["iteration"], y=agg["mean"], name=str(val),
                                line=dict(color=c, width=2), mode="lines+markers"))
    fig.add_trace(go.Scatter(
        x=pd.concat([agg["iteration"], agg["iteration"][::-1]]),
        y=pd.concat([agg["mean"]+agg["std"], (agg["mean"]-agg["std"])[::-1]]),
        fill="toself", fillcolor=c, opacity=0.15, line=dict(width=0),
        showlegend=False, hoverinfo="skip"))
fig.update_layout(title=f"{'mean_reward_this_iter'} by {'gfn_loss'}", xaxis_title="iteration",
                    plot_bgcolor="white", hovermode="x unified")

In [ ]:
print("fig.layout.margin:", fig.layout.margin)
for i, tr in enumerate(fig.data):
    print(i, tr.type, "domain=", getattr(tr, "domain", None))
    if tr.type == "parcats":
        print(" parcats dims:", [len(getattr(d, 'values', [])) for d in tr.dimensions])

fig.layout.margin: layout.Margin()
0 scatter domain= None
1 parcats domain= parcats.Domain({
    'y': [0, 0.42]
})
 parcats dims: [100, 100, 100, 100]


In [122]:
metrics.groupby(["gfn_loss", "iteration"]).agg(
    mean_reward_this_iter_mean=("mean_reward_this_iter", "mean"),
    mean_reward_this_iter_std=("mean_reward_this_iter", "std"),
)

mean_reward_this_iter_mean  \
gfn_loss          iteration                               
detailedbalance   1                            0.892174   
                  2                            0.895631   
                  3                            0.901874   
                  4                            0.891745   
                  5                            0.897352   
                  6                            0.901109   
                  7                            0.903566   
                  8                            0.905299   
                  9                            0.902775   
                  10                           0.903025   
                  11                           0.909970   
                  12                           0.907699   
                  13                           0.906055   
                  14                           0.903477   
                  15                           0.900961   
flowmatching      1                            0.891143   
                  2                            0.895737   
                  3                            0.902482   
                  4                            0.899137   
                  5                            0.901106   
                  6                            0.901750   
                  7                            0.899300   
                  8                            0.901329   
                  9                            0.901407   
                  10                           0.897690   
                  11                           0.903922   
                  12                           0.908403   
                  13                           0.910870   
                  14                           0.907486   
                  15                           0.892728   
forwardlooking    1                            0.889706   
                  2                            0.901558   
                  3                            0.901732   
                  4                            0.901407   
                  5                            0.901589   
                  6                            0.905281   
                  7                            0.903542   
                  8                            0.901989   
                  9                            0.905442   
                  10                           0.903499   
                  11                           0.906304   
                  12                           0.910748   
                  13                           0.903743   
                  14                           0.908002   
                  15                           0.903752   
trajectorybalance 1                            0.910771   
                  2                            0.916849   
                  3                            0.918841   
                  4                            0.923102   
                  5                            0.919667   
                  6                            0.922650   
                  7                            0.923799   
                  8                            0.923370   
                  9                            0.923575   
                  10                           0.924474   
                  11                           0.930137   
                  12                           0.926523   
                  13                           0.923910   
                  14                           0.927418   
                  15                           0.920876   

                             mean_reward_this_iter_std  
gfn_loss          iteration                             
detailedbalance   1                           0.033907  
                  2                           0.020939  
                  3                           0.017887  
                  4                           0.025300  
                  5         